# Environment and data check

This notebook verifies the project environment and inspects the bundled PlantVillage and PlantDoc samples. Use the **Python (plant-disease-assistant)** kernel.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from plant_disease.paths import SAMPLE_DIR

device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
print('Project:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Device:', device)

## Sample manifest

The sample is only for inspecting formats and testing code. The full datasets are downloaded with `make data`.

In [ ]:
manifest = pd.read_csv(SAMPLE_DIR / 'manifest.csv')
display(manifest.groupby('dataset').size().rename('images'))
display(manifest.head())

## PlantVillage examples

In [ ]:
pv_rows = manifest[manifest.dataset == 'PlantVillage'].groupby('labels').head(1)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for axis, (_, row) in zip(axes.flat, pv_rows.iterrows()):
    image = Image.open(SAMPLE_DIR / row.file).convert('RGB')
    axis.imshow(image)
    axis.set_title(row.labels.replace('___', '\n'), fontsize=9)
    axis.axis('off')
plt.tight_layout()

## PlantDoc annotations

PlantDoc stores bounding boxes as `[x, y, width, height]`. The extraction script also writes normalized YOLO labels.

In [ ]:
records = json.loads((SAMPLE_DIR / 'PlantDoc' / 'annotations.json').read_text())
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for axis, record in zip(axes.flat, records[:6]):
    image = Image.open(SAMPLE_DIR / 'PlantDoc' / record['file']).convert('RGB')
    draw = ImageDraw.Draw(image)
    for annotation in record['annotations']:
        x, y, width, height = annotation['bbox_xywh']
        draw.rectangle((x, y, x + width, y + height), outline='red', width=3)
    axis.imshow(image)
    axis.set_title(record['annotations'][0]['label'], fontsize=9)
    axis.axis('off')
plt.tight_layout()

## Next steps

1. Download the complete datasets.
2. Reproduce the duplicate and annotation audit.
3. Freeze a deduplicated PlantDoc field test split.
4. Implement the PlantVillage-only classification baseline before dataset adaptation.